In [ ]:
# =============================================================================
# Experiment 9: Email Spam Classification using Text Classification (Naive Bayes & SVM)
# Objective: Implement a text classification algorithm (e.g., Naive Bayes or SVM) to classify emails as spam or not spam. [CO4]
# =============================================================================

# 1. Import Required Libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [ ]:
# =============================================================================
# 2. Create / Load Email Text Dataset (Ham vs. Spam)
# Label: 0 = Not Spam (Ham), 1 = Spam
# =============================================================================

ham_emails = [
    "Hi team, please find the attached minutes from today's sprint planning meeting.",
    "Reminder: Your MCA lab record submission is scheduled for tomorrow morning at 10 AM.",
    "Hey Ajay, let's review the FastAPI database schema changes before deploying to staging.",
    "Your monthly broadband invoice for November has been generated and paid via UPI.",
    "Can we reschedule our 3 PM sync to 4:30 PM? Let me know if that works for you.",
    "Thank you for registering for the cybersecurity workshop. Here is your joining link.",
    "Please review the pull request for the ThreatMapper backend authentication module.",
    "Your order has been shipped and will arrive by Friday evening. Track your package online.",
    "Attached is the updated project timeline and resource allocation spreadsheet.",
    "Good morning, kindly share the notes from yesterday's Information Security lecture."
] * 30

spam_emails = [
    "CONGRATULATIONS! You have won a FREE iPhone 16 Pro! Click here to claim your prize NOW!",
    "URGENT: Your bank account has been suspended. Verify your KYC details immediately to avoid closure.",
    "Make $5,000 a week working from home! No experience needed. Sign up today for instant cash!",
    "LIMITED TIME OFFER: Get 90% OFF on luxury watches and medications. Order now before stock runs out!",
    "You have an unclaimed lottery reward of Rs. 25,00,000! Reply with your bank details to transfer funds.",
    "Hot singles in your area want to meet tonight! Click this exclusive link for free access.",
    "FINAL NOTICE: Your tax refund of $940 is pending. Submit your credit card info to process payment.",
    "Free crypto giveaway! Send 0.1 BTC to this wallet address and receive 1.0 BTC back instantly!",
    "Lose 15 kgs in 10 days with this miracle pill! Doctors hate this one simple trick!",
    "Alert: Unauthorized login detected. Reset your password at http://bit.ly/fake-reset-link immediately."
] * 30

texts = ham_emails + spam_emails
labels = [0] * len(ham_emails) + [1] * len(spam_emails)

df = pd.DataFrame({'Email_Text': texts, 'Label': labels})
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

print(f"Total Emails: {len(df)} | Not Spam (0): {(df['Label']==0).sum()} | Spam (1): {(df['Label']==1).sum()}")
print(df.head())

In [ ]:
# =============================================================================
# 3. Text Preprocessing & TF-IDF Feature Extraction
# Converting raw email text into numerical TF-IDF vectors (unigrams & bigrams).
# =============================================================================

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df['Email_Text'], df['Label'], test_size=0.25, random_state=42, stratify=df['Label']
)

vectorizer = TfidfVectorizer(stop_words='english', lowercase=True, ngram_range=(1, 2))
X_train_tfidf = vectorizer.fit_transform(X_train_text)
X_test_tfidf = vectorizer.transform(X_test_text)

print(f"TF-IDF Vocabulary Size: {len(vectorizer.vocabulary_)} features")

In [ ]:
# =============================================================================
# 4. Train Multinomial Naive Bayes & Support Vector Machine (LinearSVC)
# =============================================================================

# Model A: Multinomial Naive Bayes
nb_model = MultinomialNB(alpha=1.0)
nb_model.fit(X_train_tfidf, y_train)
nb_preds = nb_model.predict(X_test_tfidf)

# Model B: Linear Support Vector Machine
svm_model = LinearSVC(random_state=42)
svm_model.fit(X_train_tfidf, y_train)
svm_preds = svm_model.predict(X_test_tfidf)

print(f"Multinomial Naive Bayes Accuracy: {accuracy_score(y_test, nb_preds):.4f}")
print(f"Linear SVM Accuracy:              {accuracy_score(y_test, svm_preds):.4f}")

In [ ]:
# =============================================================================
# 5. Evaluate Models, Visualize Confusion Matrices & Test Custom Emails
# =============================================================================

print("--- Naive Bayes Classification Report ---")
print(classification_report(y_test, nb_preds, target_names=['Not Spam', 'Spam']))

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# Confusion Matrix: Naive Bayes
sns.heatmap(confusion_matrix(y_test, nb_preds), annot=True, fmt='d', cmap='Blues',
            xticklabels=['Not Spam', 'Spam'], yticklabels=['Not Spam', 'Spam'], ax=axes[0])
axes[0].set_title('Multinomial Naive Bayes Confusion Matrix')
axes[0].set_xlabel('Predicted')
axes[0].set_ylabel('Actual')

# Confusion Matrix: Linear SVM
sns.heatmap(confusion_matrix(y_test, svm_preds), annot=True, fmt='d', cmap='Greens',
            xticklabels=['Not Spam', 'Spam'], yticklabels=['Not Spam', 'Spam'], ax=axes[1])
axes[1].set_title('Linear SVM Confusion Matrix')
axes[1].set_xlabel('Predicted')
axes[1].set_ylabel('Actual')

plt.tight_layout()
plt.show()

# Test on unseen sample emails
new_emails = [
    "Hey, can you send over the PostgreSQL connection string for the test server?",
    "WINNER! Claim your free cash reward of $10,000 today by clicking this urgent link!"
]
new_tfidf = vectorizer.transform(new_emails)
nb_new_preds = nb_model.predict(new_tfidf)

print("--- Sample Email Predictions (Naive Bayes) ---")
for email, pred in zip(new_emails, nb_new_preds):
    print(f"[{'SPAM' if pred == 1 else 'NOT SPAM'}] -> {email}")